# AI Personal Finance Agent — Colab Pipeline

This notebook is the **cloud compute environment** for the project. It does not
replace your local Mac + GitHub repo — it *syncs with it*. Read
`PROJECT_TRACKER.md` in the repo for full project context before continuing.

**How this fits together (read this once):**

| Where | What happens there |
|---|---|
| **Your Mac (terminal)** | `git` is the source of truth: init the repo, commit, push, pull. Light local editing if you want (e.g. in VS Code). No heavy compute needed. |
| **GitHub** | The sync point between your Mac and Colab. Nothing "lives" only in Colab — anything worth keeping gets pushed back here before the Colab runtime is closed. |
| **Google Colab (this notebook)** | Heavy / iterative compute: LLM calls for categorization and the agent layer, analytics over larger datasets, experimentation. Colab's disk is **ephemeral** — it is wiped when the runtime disconnects, so this notebook always starts with a fresh `git clone`/`git pull` and ends with a `git push`. |

If you skip the terminal step and never push, work done in Colab disappears
the moment the runtime recycles (Colab free tier disconnects after ~90 min
idle, or 12h max). Always push before you close the tab.


## Step 0 — One-time setup on your Mac (do this in Terminal, not here)

Do this once, before touching Colab, using the Phase 1 scaffold you already
unzipped.

```bash
cd ~/path/to/ai-finance-agent      # wherever you unzipped it

# If git isn't initialized yet (the zip intentionally excludes .git):
git init
git add -A
git commit -m "Phase 1: repository scaffold"
git branch -M main

# Create the GitHub repo (pick ONE of these two ways):
#   A) Using the GitHub CLI (fastest, if you have `gh` installed):
gh repo create ai-finance-agent --private --source=. --remote=origin --push

#   B) Manually: create an empty repo at https://github.com/new named
#      ai-finance-agent (do NOT initialize it with a README), then:
git remote add origin https://github.com/<your-username>/ai-finance-agent.git
git push -u origin main
```

Once this is done, come back here — the rest of this notebook talks to that
GitHub repo, not to files on your Mac (Colab cannot see your Mac's disk).


In [ ]:
#@title Step 1 — Configure repo details
GITHUB_USERNAME = "your-github-username"  #@param {type:"string"}
REPO_NAME = "ai-finance-agent"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
PRIVATE_REPO = True  #@param {type:"boolean"}

REPO_URL_HTTPS = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
print("Repo:", REPO_URL_HTTPS)


## Step 2 — Authenticate (only needed for a private repo)

If your repo is **public**, skip this cell — anonymous `git clone` works fine.

If it's **private**, you need a GitHub Personal Access Token (fine-grained,
scoped to just this repo, "Contents: Read and write") so Colab can clone and
push. **Never hardcode the token in a cell.** Two safe options:

1. **Recommended:** click the key icon (🔑) in the left sidebar of Colab →
   "Secrets" → add a secret named `GITHUB_TOKEN` → toggle "Notebook access"
   on. The cell below reads it from there.
2. If Secrets isn't available, the cell falls back to a hidden `getpass`
   prompt so the token is never printed or saved in the notebook file.


In [ ]:
import subprocess

token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass

if not token and PRIVATE_REPO:
    from getpass import getpass
    token = getpass("Enter your GitHub Personal Access Token (input hidden): ")

if PRIVATE_REPO and token:
    AUTH_REPO_URL = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
else:
    AUTH_REPO_URL = REPO_URL_HTTPS

print("Auth configured." if PRIVATE_REPO else "Public repo — no auth needed.")


In [ ]:
#@title Step 3 — Clone (fresh runtime) or pull (already cloned)
import os

REPO_DIR = f"/content/{REPO_NAME}"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {AUTH_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin {BRANCH}

%cd {REPO_DIR}
!git config user.email "harshith.gundra@gmail.com"
!git config user.name "Harshith Reddy"
!ls -la


In [ ]:
#@title Step 4 — Install dependencies
!pip install -q -r requirements.txt
print("Dependencies installed.")


In [ ]:
#@title Step 5 — Sanity check (mirrors the Phase 1 local validation)
!pytest -q


## Step 6 — Saving work back to GitHub

Run the helper cell below **any time you want to checkpoint progress**, and
always before you close the notebook or let the runtime idle out. Colab does
not autosave your repo state anywhere except GitHub.


In [ ]:
#@title Helper: commit & push current state back to GitHub
def save_progress(commit_message: str):
    import subprocess
    os.chdir(REPO_DIR)
    subprocess.run(["git", "add", "-A"], check=True)
    result = subprocess.run(["git", "commit", "-m", commit_message], capture_output=True, text=True)
    print(result.stdout, result.stderr)
    if result.returncode == 0:
        push = subprocess.run(["git", "push", AUTH_REPO_URL, BRANCH], capture_output=True, text=True)
        print(push.stdout, push.stderr)
    else:
        print("Nothing to commit (working tree clean), skipping push.")

# Example usage once a phase produces new files:
# save_progress("Phase 2: add mock transaction generator + sample CSV")


## What runs where, in practice

- **Phase 1 (repo setup):** done, on your Mac.
- **Phase 2 (mock data generation):** light CPU work — fine either place;
  we'll run it here in Colab going forward since that's your preference,
  and push the resulting `data/mock_transactions.csv` back to GitHub.
- **Phase 3 (database):** built here from the CSV each session — the
  `.db` file is **not** committed (see `.gitignore`), it's a rebuildable
  artifact, so there's nothing to lose when the Colab runtime resets.
- **Phase 4 & 7 (LLM categorization + agent):** the heavy-compute phases —
  this is where Colab's free GPU matters most, especially if we run an
  open-source model locally-in-Colab (e.g. via `transformers` +
  4-bit quantization) instead of calling a hosted API.
- **Phase 5–6 (analytics, anomaly detection):** here, since it chains
  directly off Phase 3/4 outputs already in this session.
- **Phase 8 (Streamlit/CLI interface):** back on your Mac — a UI you
  interact with locally isn't a great fit for a Colab runtime.
- **Phase 9:** this notebook *is* Phase 9's deliverable, evolving as we go.
- **Phase 10 (Plaid Sandbox):** either place; we'll decide when we get there.

Next: Phase 2 will add cells below this line to generate the mock
transaction dataset.
